# About this notebook

**Prepared by:** Maryam Hiradfar  
MD–PhD Candidate, Harvard Medical School & Harvard Biophysics  
Catana Lab | Martinos Center for Biomedical Imaging

This notebook serves as an introduction to this  this repository. The goal is to familiarize new lab members with both the PET concepts and the software architecture before contributing to the codebase.

Last updated: June 17 2026

##  Setup

Run this notebook from the root of the repository, so that imports like `tracers.fdg`, `sim.tac_simulator`, and `kinetics.models.twotcm` work correctly.

If imports fail, first check that:

- the notebook is running inside the correct Python environment,
- the repository root is on `sys.path`,
- the repo has been installed with `pip install -e .` if you are using editable installation.

In [1]:
from pathlib import Path
import sys

# Optional: add repository root manually if needed.
# If this notebook is placed inside notebooks/, this points one level up.
repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repository root:", repo_root)

Repository root: /Users/maryamhiradfar/Documents/GitHub/dual_tracer_sims


In [4]:
import numpy as np
import matplotlib.pyplot as plt

# Core tracer objects
from tracers.fdg import FDGTracer
from tracers.pbr28 import PBR28Tracer

# Kinetic simulation
from kinetics.models.twotcm import simulate_2tcm

# Frame averaging and noise
from utils.frame_average import frame_average
from core.noise import add_noise_voxel

# Optional separation pieces used later
from scipy.optimize import nnls

## How the repository is organized: OOP architecture

Before running simulations, it is helpful to understand the overall design of the codebase.

This repository is organized around a few interchangeable object types:

1. **Tracer objects** describe tracer-specific biology and physics.
2. **Library builders** generate basis functions/dictionaries.
3. **Separation algorithms** estimate the individual tracer curves from a mixed dual-tracer TAC.
4. **Simulation functions** connect tracers, frame schedules, noise, and algorithms.

Since this structure is modular, we should be able to swap FDG for PBR28, a gamma basis for a kinetic basis, or NNLS for a TCN without rewriting the whole pipeline.


### Big-picture flow

1. Tracer object generation
2. Using the tracer object to generate its AIF and kinetic parameters
3. Feeding the above to a 2TCM model for kinetic modeling
4. Physical decay and noise: So far, we have not considered effects from radioactive decay and noise. At this stage, we can add these components
5. Simulating dual-tracer TAC: sum the measured-domain TACs from the two tracers to generate the dual-tracer signal. Since PET measures total detected activity, the scanner does not inherently know which tracer generated each event.
4. Frame-averaging: real PET data are not instantaneous measurements but averages over acquisition frames. Therefore, we average the continuous simulated signal over the same frame schedule used in PET acquisition.
7. Deconstruction into single TAC measured curves: (these separated TACs are still in the measured domain and therefore still contain decay effects).
8. Decay correction: we would want to know the biology -- estimate the underlying biological tracer concentration from the measured activity. Later time points require larger correction factors because fewer radioactive nuclei remain available to emit positrons, even if the biological tracer concentration is still present in tissue.

Code feautre: The simulation code should not need to know all the details of each tracer or each algorithm. It should only rely on shared interfaces.

### What is an interface?

An **interface** defines what methods an object must provide.

For example, every tracer in this repository should provide:

```python
tracer.aif(t_abs, delta_min=0.0)
tracer.true_params()
```

That means downstream code can use any tracer object as long as it follows this pattern.

This is the core OOP idea being used here: different objects can be used through the same shared interface.

### Tracer classes

The base tracer class defines the attributes, methods and expected behavior of the class:

```python
@dataclass
class Tracer:
    name: str
    half_life_min: float

    def aif(self, t_abs, delta_min=0.0):
        raise NotImplementedError

    def true_params(self):
        raise NotImplementedError
```

Concrete tracer classes such as `FDGTracer` and `PBR28Tracer` implement those methods.

The important point is that the simulator does not need separate code paths like:

```python
if tracer == "FDG":
    ...
elif tracer == "PBR28":
    ...
```

Instead, it can simply call:

```python
Cp = tracer.aif(t)
params = tracer.true_params()
```

Before being able to do so however, we first need to create the tracers by instantiating the tracer objects: 

In [6]:

pbr = PBR28Tracer(name="PBR28", half_life_min=20.4, scale=1.0)
fdg = FDGTracer(name="FDG", half_life_min=109.8, scale=1.0)
tracers = [pbr, fdg]


    
    

We can now inspect several "attributes" of our tracer objects, such as their name, half-life, and other stored properties.

In [7]:
for tracer in tracers:
    print("Tracer:", tracer.name)
    print("Half-life:", tracer.half_life_min, "min")
    print("Kinetic parameters:", tracer.true_params())

Tracer: PBR28
Half-life: 20.4 min
Kinetic parameters: {'K1': 0.18, 'k2': 0.25, 'k3': 0.04, 'k4': 0.01}
Tracer: FDG
Half-life: 109.8 min
Kinetic parameters: {'K1': 0.12, 'k2': 0.25, 'k3': 0.06, 'k4': 0.0}


### Configuration dataclasses

Many parts of the repository use small configuration dataclasses.

For example, a gamma basis config might store:

```python
n_t0
n_tau
tau_min
tau_max
scale
```

This is cleaner than passing many loose arguments into every function.

Benefits:

- easier to read,
- easier to save and reproduce,
- easier to validate,
- easier to pass around as one object.

### Library builders

A **library** is a matrix of candidate basis TACs.

For example, a gamma library contains many gamma-shaped curves with different onset times and time constants.

The shared interface is:

```python
class LibraryBuilder:
    def build_library(self, t):
        ...
```

Each builder returns a standard result:

```python
LibraryResult(lib=lib, Gamma=Gamma)
```

where:

- `lib` has shape `(n_basis, n_time_points)`,
- `Gamma` usually has shape `(n_time_points, n_basis)`.

This makes it possible to compare several basis designs while keeping the separation code mostly unchanged.

### Registry pattern

The repository also uses a **registry pattern**.

Instead of writing a large block of conditional logic like:

```python
if name == "uniform_gamma":
    builder = UniformGammaBuilder(config)
elif name == "injection_centered_gamma":
    builder = InjectionCenteredGammaBuilder(config)
```

classes can register themselves:

```python
@register
class InjectionCenteredGammaBuilder(LibraryBuilder):
    name = "injection_centered_gamma"
```

Then the object can be created by name:

```python
builder = create("injection_centered_gamma", config)
```

This makes the repository more plugin-like. Adding a new method does not require changing a central if/else block.

### Separation algorithms

Separation algorithms also follow a shared interface:

```python
class SeparationAlgorithm:
    def separate(self, y, t_frames, Phi_1, Phi_2):
        ...
```

Each method returns a `SeparationResult` containing:

```python
tracer1_curve
tracer2_curve
coef_1
coef_2
metadata
```

This means the simulator can treat joint NNLS, sequential NNLS, recursive NNLS, and eventually TCN-based methods in a common way.

### Why this design matters for this project

This design is useful because the scientific questions will evolve.

For example, later we might want to add:

```python
class FEBEOVTracer(Tracer):
    ...
```

or:

```python
class KineticInformedLibraryBuilder(LibraryBuilder):
    ...
```

or:

```python
class TCNLibrarySeparation(SeparationAlgorithm): # This is what you will build yay!!
    
    ...
```

If each component follows the expected interface, we can add new tracers, basis libraries, and algorithms without rewriting the rest of the codebase.

### Exercise: trace the data flow

Before continuing, trace what happens in a single simulation -- run_single_experiment.py is a good place to start!

Answer these questions:

1. Where does the AIF come from?
2. Where do the 2TCM parameters come from?
3. Where is the smooth TAC converted into frame averages?
4. Where is radioactive decay applied?
5. Where is noise added?
6. What does the separation algorithm receive as input?
7. What does it return?

A good way to understand this repository is to draw the data flow from `Tracer` → `simulate_2tcm` → `frame_average` → noisy dual TAC → basis matrices → separated TACs.